# 1D Quantum Harmonic Oscillator: Finite-Difference Hamiltonian

This notebook calculates the approximate second derivative of the quantum harmonic oscillator and constructs a hamiltonian matrix:
Everything here is kept in **one spatial dimension**,
We use dimensionless units, $\hbar = m = \omega = 1$, so the Hamiltonian is
$$
H = -\frac{1}{2}\frac{d^2}{dx^2} + \frac{1}{2}x^2 ,
$$
and the known analytic energies are $E_n = n + \tfrac{1}{2}$. These are the
same conventions used throughout the project.

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.sparse import diags
from scipy.sparse.linalg import eigsh
from scipy.special import eval_hermite, gammaln
pd.set_option("display.precision", 10)

## Shared grid and parameter definitions

- Physical units: dimensionless, $\hbar = m = \omega = 1$
- Spatial interval: $x \in [-L, L]$ with $L = 8$
- Number of grid points: $N = 1000$, including the two boundary points
- Boundary condition: Dirichlet, $\psi(\pm L) = 0$
- Requested eigenstates: the lowest 6

In [3]:
# Dimensionless units
hbar = 1.0
mass = 1.0
omega = 1.0
# Shared numerical-box parameters
L = 8.0
N = 1000
num_states = 6
# Full grid, including the two boundary points
x_full = np.linspace(-L, L, N)
dx = x_full[1] - x_full[0]
# Dirichlet boundary conditions fix psi(-L) = psi(L) = 0, so only the interior points are unknowns in the eigenvalue problem.
x = x_full[1:-1]
M = x.size
print(f"Domain: [{-L}, {L}]")
print(f"Total grid points N: {N}")
print(f"Interior unknowns M: {M}")
print(f"Grid spacing dx: {dx:.6f}")

Domain: [-8.0, 8.0]
Total grid points N: 1000
Interior unknowns M: 998
Grid spacing dx: 0.016016


## Finite-difference second derivative

For an interior grid point $x_i$, the centered, second-order finite
difference approximation to the second derivative is
$$
\left.\frac{d^2\psi}{dx^2}\right|_{x_i} \approx
\frac{\psi_{i+1} - 2\psi_i + \psi_{i-1}}{\Delta x^2}.
$$

Applied at every interior point, this turns the derivative into a
tridiagonal matrix with $-2$ on the main diagonal and $1$ on the first
off-diagonals (before dividing by $\Delta x^2$).

To make the pattern easy to check by eye, we first build this matrix on a
tiny 6-point grid before using it at full resolution.

In [4]:
# Small demonstration grid
M_demo = 6
lower_demo = np.ones(M_demo - 1)
main_demo = -2.0 * np.ones(M_demo)
upper_demo = np.ones(M_demo - 1)
D2_demo = diags(
    [lower_demo, main_demo, upper_demo], [-1, 0, 1], shape=(M_demo, M_demo)
).toarray()
print("Unscaled second-derivative pattern (before dividing by dx^2):")
print(D2_demo)
assert D2_demo.shape == (M_demo, M_demo), "unexpected shape"
assert np.allclose(D2_demo, D2_demo.T), "pattern is not symmetric"
assert np.allclose(np.diag(D2_demo), -2.0), "main diagonal should be -2"
print("\nShape, symmetry, and diagonal value all check out.")

Unscaled second-derivative pattern (before dividing by dx^2):
[[-2.  1.  0.  0.  0.  0.]
 [ 1. -2.  1.  0.  0.  0.]
 [ 0.  1. -2.  1.  0.  0.]
 [ 0.  0.  1. -2.  1.  0.]
 [ 0.  0.  0.  1. -2.  1.]
 [ 0.  0.  0.  0.  1. -2.]]

Shape, symmetry, and diagonal value all check out.


In [5]:
# Full-resolution second-derivative matrix, stored as sparse CSR
lower = np.ones(M - 1)
main = -2.0 * np.ones(M)
upper = np.ones(M - 1)
D2 = diags(
    diagonals=[lower, main, upper],
    offsets=[-1, 0, 1],
    shape=(M, M),
    format="csr",
) / dx**2
print(f"D2 shape: {D2.shape}")
print(f"D2 symmetric to numerical precision: {abs(D2 - D2.T).max() < 1e-10}")
print(f"D2 stored nonzeros: {D2.nnz} (expected {3*M - 2})")

D2 shape: (998, 998)
D2 symmetric to numerical precision: True
D2 stored nonzeros: 2992 (expected 2992)


In [7]:
energies_num, states = eigsh(H, k=num_states, which="SA")

# eigsh does not guarantee ascending order; sort explicitly
order = np.argsort(energies_num)
energies_num = energies_num[order]
states = states[:, order]

energies_num

array([0.49999198, 1.49995992, 2.49989579, 3.49979959, 4.49967132,
       5.49951098])